In [18]:
###### IMPORT AND UTILITIES ######

import torch
import librosa as lr
import numpy as np
import matplotlib.pyplot as plt
import os
import scipy.io.wavfile
from IPython.display import Audio

In [19]:
###### LOADING THE AUDIOS ######

files = [f for f in os.listdir('../data/generated/') if f.endswith('wav')]
files.sort()
print(f'files = {files}\n')

amplitudes = {}
sampling_rates = {}

for i, file in enumerate(files):
    print(f'loading sample {i+1}...')
    file_path = os.path.join('../data/generated/', file)
    amp, sr = lr.load(file_path, sr = None)                
    amplitudes[file] = amp                                 
    sampling_rates[file] = sr                              
                                                           
print('\nSuccessfully loaded')
print(f'sampling rate: {sr}Hz')

files = ['sample_1_a_90s_rock_song_with.wav', 'sample_2_a_calm_relaxing_lofi.wav', 'sample_3_epic_cinematic_music.wav', 'sample_4_a_simple_melancholic.wav']

loading sample 1...
loading sample 2...
loading sample 3...
loading sample 4...

Successfully loaded
sampling rate: 32000Hz


In [20]:
###### DENOISERS LOADING ######

print('loading model "dns64" ...')
dns64 = torch.hub.load('facebookresearch/denoiser', 'dns64', pretrained = True)
dns64.eval()
print('model "dns64" loaded successfully\n')

print('loading model "master64" ...')
master64 = torch.hub.load('facebookresearch/denoiser', 'master64', pretrained = True)
master64.eval()
print('model "master64" loaded successfully')

# torch.hub.load is a pytorch function used to load models from github repositories or other online sources.
# The first argument tells it to go to the user "facebookresearch" and locate the repository "denoiser". Second argument specifies 
# which model we want to load. I'm going to try and compare two different models: 

#    ·dns64: belongs to 'dns' models, specifically optimized for the DNS challenges (Deep Noise Suppression). They're more focused
#            on speech, but the datasets used for training contain a lot of background noise.  

#    ·master64: belongs to 'master' models, trained on a very large dataset that includes music, speech and ambient sounds.
#               They are more generic models, but they are also older.

# Lastly, pretrained = True serves to download the model with his pre-trained weights and model.eval() sets the model to evaluation mode.

loading model "dns64" ...


Using cache found in /Users/azzurragiordano/.cache/torch/hub/facebookresearch_denoiser_main


model "dns64" loaded successfully

loading model "master64" ...
model "master64" loaded successfully


Using cache found in /Users/azzurragiordano/.cache/torch/hub/facebookresearch_denoiser_main


In [21]:
###### DENOISER FUNCTION ######

def denoiser_apply(model, audio):
    
    '''
    Applies a denoising model to audio.

    input: the model we want to use and generated audio's path 
    output: restored audio's path
    
    '''

    original_sr = sampling_rates[audio]
    original_amp = amplitudes[audio]

    model_sr = 16000    # The model works with sr = 16000Hz

    resampled_amp = lr.resample(original_amp, orig_sr = original_sr, target_sr = model_sr)

    amp_torch = torch.tensor(resampled_amp).unsqueeze(0).unsqueeze(0) # The model expects a three-dimensional input
                                                                      # [batch_size, channels, size], so we added two dimensions at the
                                                                      # beginning. Now amp_torch's shape is [1, 1, len(resampled_amp)].
    with torch.no_grad(): # Disables gradient computation
        restored_torch = model(amp_torch)[0]

    restored_numpy = restored_torch.squeeze().cpu().numpy()

    restored_amp = lr.resample(restored_numpy, orig_sr = model_sr, target_sr = original_sr)

    return restored_amp

In [22]:
###### DENOISING ######

restored_dir_dns64 = '../data/restored/dns64/'
restored_dir_master64 = '../data/restored/master64/'
os.makedirs(restored_dir_master64, exist_ok = True)
os.makedirs(restored_dir_dns64, exist_ok = True)

restored_amplitudes_dns64 = {}
restored_amplitudes_master64 = {}

for i, audio in enumerate(files):

    dns64_amp = denoiser_apply(dns64, audio)
    master64_amp = denoiser_apply(master64, audio)

    restored_amplitudes_dns64[audio] = dns64_amp
    restored_amplitudes_master64[audio] = master64_amp

    dns64_path = os.path.join(restored_dir_dns64, audio) 
    master64_path = os.path.join(restored_dir_master64, audio)

    sr = sampling_rates[audio]
    
    scipy.io.wavfile.write(dns64_path, rate=sr, data=dns64_amp)
    scipy.io.wavfile.write(master64_path, rate=sr, data=master64_amp)

    print(f'sample {i+1} restored succesfully with both models. Files have been saved in {dns64_path} and\n{master64_path}')
    print(f'sampling rate: {sr}Hz\n')

sample 1 restored succesfully with both models. Files have been saved in ../data/restored/dns64/sample_1_a_90s_rock_song_with.wav and
../data/restored/master64/sample_1_a_90s_rock_song_with.wav
sampling rate: 32000Hz

sample 2 restored succesfully with both models. Files have been saved in ../data/restored/dns64/sample_2_a_calm_relaxing_lofi.wav and
../data/restored/master64/sample_2_a_calm_relaxing_lofi.wav
sampling rate: 32000Hz

sample 3 restored succesfully with both models. Files have been saved in ../data/restored/dns64/sample_3_epic_cinematic_music.wav and
../data/restored/master64/sample_3_epic_cinematic_music.wav
sampling rate: 32000Hz

sample 4 restored succesfully with both models. Files have been saved in ../data/restored/dns64/sample_4_a_simple_melancholic.wav and
../data/restored/master64/sample_4_a_simple_melancholic.wav
sampling rate: 32000Hz



In [23]:
###### CREATING AUDIO PLAYER ######

print('##########################')
print('restored audios with dns64')
print('##########################')
i = 1
for file, amp in restored_amplitudes_dns64.items():
    sr = sampling_rates[file]
    print(f'\n---sample {i}---')
    display(Audio(data = amp, rate = sr))
    i += 1

print('\n#############################')
print('restored audios with master64')
print('#############################')
j = 1
for file, amp in restored_amplitudes_master64.items():
    sr = sampling_rates[file]
    print(f'\n---sample {j}---')
    display(Audio(data = amp, rate = sr))
    j += 1


##########################
restored audios with dns64
##########################

---sample 1---



---sample 2---



---sample 3---



---sample 4---



#############################
restored audios with master64
#############################

---sample 1---



---sample 2---



---sample 3---



---sample 4---


In [ ]:
'''As we can see, the audio is not restored at all! This is because I tried a pipeline based on state-of-the-art denoising models 
(dns64 and master64) which resulted in an almost complete suppression of the musical signal itself, which was mistakenly classified as noise. This 
negative outcome is, in fact, an important conclusion: it highlights the need for models specifically trained for music, such as those used 
for source separation.'''